# 05 — Normalización Clínica

**Parte 1 — Condiciones** (`clinical_trials.parquet`)
- Parseo de `conditions` y `condition_meshes` (almacenados como JSON strings)
- Extracción de MeSH ids y términos canónicos
- Filtrado de keywords metodológicos
- Construcción de `condition_text`: campo unificado para el filtro semántico del pipeline online

**Parte 2 — Criterios de elegibilidad** (`04_eligibility_parsed_llm.parquet`)
- Normalización de símbolos (`>=` → `≥`, espaciado)
- Expansión de abreviaturas médicas frecuentes
- Tagging de entidades con scispaCy (`[condition]`, `[medication]`, `[lab value]`, `[demographics]`)

**Outputs:**
- `data/05_clinical_trials_normalized.parquet`
- `data/05_eligibility_normalized.parquet`


## 0. Imports y constantes

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
import spacy

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR            = Path('..') / 'data'
PARQUET_TRIALS      = DATA_DIR / 'clinical_trials.parquet'
PARQUET_ELIGIBILITY = DATA_DIR / '04_eligibility_parsed_llm.parquet'
PARQUET_OUT_TRIALS  = DATA_DIR / '05_clinical_trials_normalized.parquet'
PARQUET_OUT_ELIG    = DATA_DIR / '05_eligibility_normalized.parquet'

# ── scispaCy ───────────────────────────────────────────────────────────────
SCISPACY_MODEL = 'en_core_sci_sm'

# ── Keywords metodológicos a excluir del condition_text ────────────────────
METHODOLOGICAL_KEYWORDS = {
    'randomized', 'randomised', 'randomized controlled trial', 'rct',
    'phase 1', 'phase 2', 'phase 3', 'phase 4',
    'phase i', 'phase ii', 'phase iii', 'phase iv',
    'multicenter', 'multicentre', 'open label', 'open-label',
    'placebo', 'double blind', 'double-blind', 'single blind',
    'crossover', 'pilot study', 'feasibility', 'safety study',
    'dose escalation', 'dose-escalation',
}


---
## Parte 1 — Normalización de condiciones

### 1.1 Carga del dataset

In [ ]:
df_trials = pd.read_parquet(PARQUET_TRIALS)
print(f'Estudios cargados: {len(df_trials):,}')
print('Columnas:', df_trials.columns.tolist())
df_trials[['nct_id', 'conditions', 'condition_meshes', 'keywords', 'overall_status', 'min_age', 'std_ages']].head(3)


### 1.2 Funciones de parseo de campos JSON

In [ ]:
def parse_json_list(value: str | list | None) -> list:
    """Parsea un campo almacenado como JSON string o devuelve la lista directamente."""
    if value is None:
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = json.loads(value)
        return parsed if isinstance(parsed, list) else []
    except (json.JSONDecodeError, TypeError):
        return []


def extract_mesh(condition_meshes_raw: str | list | None) -> tuple[list[str], list[str]]:
    """Extrae mesh_ids y mesh_terms desde el campo condition_meshes."""
    entries = parse_json_list(condition_meshes_raw)
    mesh_ids   = [e['id']   for e in entries if isinstance(e, dict) and 'id'   in e]
    mesh_terms = [e['term'] for e in entries if isinstance(e, dict) and 'term' in e]
    return mesh_ids, mesh_terms


def normalize_conditions(conditions_raw: str | list | None) -> list[str]:
    """Normaliza condiciones: parsea JSON, lowercase, strip, dedup."""
    items = parse_json_list(conditions_raw)
    seen, result = set(), []
    for item in items:
        normalized = str(item).lower().strip()
        if normalized and normalized not in seen:
            seen.add(normalized)
            result.append(normalized)
    return result


def filter_keywords(keywords_raw: str | list | None) -> list[str]:
    """Filtra keywords metodológicos, retorna solo los clínicamente relevantes."""
    items = parse_json_list(keywords_raw)
    return [
        str(item).lower().strip() for item in items
        if str(item).lower().strip() not in METHODOLOGICAL_KEYWORDS
    ]


### 1.3 Construcción de `condition_text`

Campo unificado para el filtro semántico del pipeline online. Combina:
- `mesh_terms` (términos canónicos MeSH, disponibles en ~88% de estudios)
- `conditions_raw` (texto libre normalizado, complementario o fallback)
- `keywords_filtered` (solo términos clínicos, no metodológicos)

Formato: `"Breast Neoplasms | Breast Cancer | HER2-positive disease"`


In [ ]:
def build_condition_text(
    mesh_terms: list[str],
    conditions_raw: list[str],
    keywords_filtered: list[str],
) -> str:
    """Construye el campo unificado para filtro semántico por condición.

    Prioridad: mesh_terms > conditions_raw > keywords_filtered.
    Deduplicación case-insensitive.
    """
    seen_lower: set[str] = set()
    parts: list[str] = []

    def add_if_new(term: str) -> None:
        key = term.lower().strip()
        if key and key not in seen_lower:
            seen_lower.add(key)
            parts.append(term.strip())

    for t in mesh_terms:         add_if_new(t)
    for t in conditions_raw:     add_if_new(t)
    for t in keywords_filtered:  add_if_new(t)

    return ' | '.join(parts)


### 1.4 Pipeline

In [ ]:
mesh_extracted    = df_trials['condition_meshes'].apply(extract_mesh)
conditions_norm   = df_trials['conditions'].apply(normalize_conditions)
keywords_filtered = df_trials['keywords'].apply(filter_keywords)

df_out = pd.DataFrame({
    'nct_id':         df_trials['nct_id'],
    'mesh_ids':       mesh_extracted.apply(lambda x: x[0]),
    'mesh_terms':     mesh_extracted.apply(lambda x: x[1]),
    'conditions_raw': conditions_norm,
    'overall_status': df_trials['overall_status'],
    'min_age':        df_trials['min_age'],
    'std_ages':       df_trials['std_ages'].apply(parse_json_list),
})

df_out['condition_text'] = [
    build_condition_text(mesh, cond, kw)
    for mesh, cond, kw in zip(df_out['mesh_terms'], df_out['conditions_raw'], keywords_filtered)
]

print(f'Estudios procesados: {len(df_out):,}')
print(f'Con MeSH:            {df_out["mesh_ids"].apply(len).gt(0).sum():,} ({df_out["mesh_ids"].apply(len).gt(0).mean():.1%})')
print(f'Sin MeSH (fallback): {df_out["mesh_ids"].apply(len).eq(0).sum():,}')
print(f'condition_text vacío:{df_out["condition_text"].eq("").sum():,}')
df_out[['nct_id', 'mesh_terms', 'conditions_raw', 'condition_text']].head(5)


### 1.5 Inspección de ejemplos

In [ ]:
print('=== Con MeSH ===')
for _, row in df_out[df_out['mesh_ids'].apply(len) > 0].head(3).iterrows():
    print(f"  [{row['nct_id']}] {row['condition_text'][:120]}")

print()
print('=== Sin MeSH (fallback a conditions_raw) ===')
for _, row in df_out[df_out['mesh_ids'].apply(len) == 0].head(3).iterrows():
    print(f"  [{row['nct_id']}] {row['condition_text'][:120]}")


### 1.6 Guardado

In [ ]:
OUTPUT_COLS_TRIALS = [
    'nct_id', 'mesh_ids', 'mesh_terms', 'conditions_raw',
    'condition_text', 'overall_status', 'min_age', 'std_ages',
]

df_out[OUTPUT_COLS_TRIALS].to_parquet(PARQUET_OUT_TRIALS, index=False)
print(f'Guardado: {PARQUET_OUT_TRIALS}')
print(f'Filas: {len(df_out):,} | Columnas: {len(OUTPUT_COLS_TRIALS)}')


---
## Parte 2 — Enriquecimiento de criterios de elegibilidad

### 2.1 Carga del dataset de elegibilidad

In [ ]:
df_elig = pd.read_parquet(PARQUET_ELIGIBILITY)
print(f'Estudios cargados: {len(df_elig):,}')
print(f'parse_status:\n{df_elig["parse_status"].value_counts()}')
df_elig.head(3)


### 2.2 Carga del modelo scispaCy

Usar `en_core_sci_sm` (liviano, ~11 MB). Si no está instalado:
```bash
pip install scispacy
pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
```


In [ ]:
nlp = spacy.load(SCISPACY_MODEL, disable=['parser', 'tagger'])
print(f'Modelo cargado: {SCISPACY_MODEL}')
print(f'Componentes activos: {nlp.pipe_names}')


### 2.3 Diccionario de abreviaturas médicas

In [ ]:
# Orden importa: expansiones más largas primero para evitar matches parciales
ABBREV_DICT: dict[str, str] = {
    # Lab values
    'eGFR':    'estimated glomerular filtration rate',
    'GFR':     'glomerular filtration rate',
    'HbA1c':   'glycated hemoglobin',
    'HgbA1c':  'glycated hemoglobin',
    'ALT':     'alanine aminotransferase',
    'AST':     'aspartate aminotransferase',
    'SGOT':    'aspartate aminotransferase',
    'SGPT':    'alanine aminotransferase',
    'ALP':     'alkaline phosphatase',
    'WBC':     'white blood cell count',
    'ANC':     'absolute neutrophil count',
    'RBC':     'red blood cell count',
    'Hgb':     'hemoglobin',
    'Hb':      'hemoglobin',
    'PLT':     'platelet count',
    'INR':     'international normalized ratio',
    'PTT':     'partial thromboplastin time',
    'APTT':    'activated partial thromboplastin time',
    'PSA':     'prostate-specific antigen',
    'CrCl':    'creatinine clearance',
    'SCr':     'serum creatinine',
    'ULN':     'upper limit of normal',
    'UNL':     'upper limit of normal',
    # Condiciones
    'T2DM':    'type 2 diabetes mellitus',
    'T2D':     'type 2 diabetes',
    'T1DM':    'type 1 diabetes mellitus',
    'T1D':     'type 1 diabetes',
    'DM':      'diabetes mellitus',
    'CKD':     'chronic kidney disease',
    'ESRD':    'end-stage renal disease',
    'CVD':     'cardiovascular disease',
    'CAD':     'coronary artery disease',
    'CHF':     'congestive heart failure',
    'HF':      'heart failure',
    'COPD':    'chronic obstructive pulmonary disease',
    'CNS':     'central nervous system',
    'NSCLC':   'non-small cell lung cancer',
    'SCLC':    'small cell lung cancer',
    'AML':     'acute myeloid leukemia',
    'CRC':     'colorectal cancer',
    'HCC':     'hepatocellular carcinoma',
    'RCC':     'renal cell carcinoma',
    'NHL':     'non-Hodgkin lymphoma',
    'HER2':    'human epidermal growth factor receptor 2',
    'EGFR':    'epidermal growth factor receptor',
    'MS':      'multiple sclerosis',
    'RA':      'rheumatoid arthritis',
    'SLE':     'systemic lupus erythematosus',
    'IBD':     'inflammatory bowel disease',
    'UC':      'ulcerative colitis',
    'HIV':     'human immunodeficiency virus',
    'HCV':     'hepatitis C virus',
    'HBV':     'hepatitis B virus',
    # Procedimientos / evaluaciones
    'RECIST':  'Response Evaluation Criteria in Solid Tumors',
    'CTCAE':   'Common Terminology Criteria for Adverse Events',
    'IV':      'intravenous',
    # Scores / índices clínicos
    'ECOG PS': 'ECOG performance status',
    'KPS':     'Karnofsky performance status',
    'LVEF':    'left ventricular ejection fraction',
    'BMI':     'body mass index',
    'BSA':     'body surface area',
    'NYHA':    'New York Heart Association functional class',
    'MELD':    'model for end-stage liver disease score',
    'MMSE':    'mini-mental state examination',
    'CDAI':    'Crohn disease activity index',
}

# Pre-compilar regex con word boundaries, ordenadas por longitud desc
# para evitar que 'ECOG' matchee antes que 'ECOG PS'
_ABBREV_PATTERNS: list[tuple[re.Pattern, str]] = [
    (re.compile(rf'\b{re.escape(abbr)}\b'), expansion)
    for abbr, expansion in sorted(ABBREV_DICT.items(), key=lambda x: -len(x[0]))
]

print(f'Abreviaturas cargadas: {len(ABBREV_DICT)}')


### 2.4 Funciones de normalización

In [ ]:
_SYMBOL_MAP: list[tuple[re.Pattern, str]] = [
    (re.compile(r'>=|=>|⩾|≧'), '≥'),
    (re.compile(r'<=|=<|⩽|≦'), '≤'),
    (re.compile(r'!=|≠'),       '≠'),
    (re.compile(r'~=|≈'),       '≈'),
]

# Espaciado: "eGFR≥45" → "eGFR ≥ 45"
_SPACING_RE = re.compile(r'([a-zA-Z0-9])([≥≤≠≈<>])([a-zA-Z0-9])')


def normalize_symbols(text: str) -> str:
    """Unifica variantes de símbolos a forma unicode canónica y normaliza espaciado."""
    for pattern, replacement in _SYMBOL_MAP:
        text = pattern.sub(replacement, text)
    text = _SPACING_RE.sub(r'\1 \2 \3', text)
    return text


def expand_abbreviations(text: str) -> str:
    """Reemplaza abreviaturas médicas por su forma expandida usando word boundaries."""
    for pattern, expansion in _ABBREV_PATTERNS:
        text = pattern.sub(expansion, text)
    return text


In [ ]:
# Patrones regex para lab values y demographics
_LAB_VALUE_RE = re.compile(
    r'\b\d+\.?\d*\s*'
    r'(?:mg/dL|g/dL|g/L|mmol/L|mL/min|mL/min/1\.73\s*m[²2]'
    r'|ng/mL|µg/mL|ug/mL|IU/L|U/L|mmHg|msec|mm|cm|%'
    r'|x10[⁹9]/L|cells/mm[³3]|copies/mL'
    r'|[xX×]\s*(?:ULN|upper\s+limit\s+of\s+normal)'
    r')(?=\s|,|;|\)|$)',
    re.IGNORECASE,
)
_DEMOGRAPHICS_RE = re.compile(
    r'\b(?:age|aged|male|female|men|women|gender|sex'
    r'|years?\s+old|adults?|children?|pediatric|elderly'
    r'|postmenopausal|premenopausal)\b',
    re.IGNORECASE,
)


def tag_entities(text: str, doc) -> str:
    """Agrega tags semánticos al final del texto basado en NER y patrones regex.

    Los tags se agregan al final (no inline) para no romper el tokenizado del embedding.
    """
    tags: set[str] = set()

    for ent in doc.ents:
        if ent.label_ == 'DISEASE':
            tags.add('[condition]')
        elif ent.label_ in ('CHEMICAL', 'DRUG'):
            tags.add('[medication]')

    if _LAB_VALUE_RE.search(text):
        tags.add('[lab value]')
    if _DEMOGRAPHICS_RE.search(text):
        tags.add('[demographics]')

    if tags:
        return text + ' ' + ' '.join(sorted(tags))
    return text


### 2.5 Verificación de las funciones sobre ejemplos

In [ ]:
test_cases = [
    'eGFR>=45mL/min/1.73m2',
    'HbA1c <= 9% and BMI < 40 kg/m2',
    'ALT <= 2.5 x ULN',
    'AST <= 3 × ULN',
    'ECOG PS 0 or 1',
    'T2DM with CKD stage 3',
    'age >= 18 years old',
    'women of childbearing potential',
    'postmenopausal females only',
    'diagnosis of NSCLC confirmed histologically',
    'platelet count >= 100 x10⁹/L',
    'QTc interval =< 480 msec',
    'tumor size <= 3 cm or <= 30 mm',
    'BP < 140 mmHg',
    'hemoglobin >= 10 g/L',
    'ANC >= 1500 cells/mm3',
    'HCV or HBV positive',
    'CNS metastases excluded',
]

print('=== Verificación pipeline de normalización ===\n')
for text in test_cases:
    step1 = normalize_symbols(text)
    step2 = expand_abbreviations(step1)
    doc   = nlp(step2)
    step3 = tag_entities(step2, doc)
    print(f'Original : {text}')
    print(f'Resultado: {step3}')
    print()


### 2.6 Pipeline de enriquecimiento

In [ ]:
def enrich_criteria_list(criteria: list[str]) -> list[str]:
    """Enriquece una lista de criterios: símbolos → abreviaturas → NER tags."""
    if not criteria:
        return []
    pre_expanded = [expand_abbreviations(normalize_symbols(c)) for c in criteria]
    docs = list(nlp.pipe(pre_expanded))
    return [tag_entities(text, doc) for text, doc in zip(pre_expanded, docs)]


In [ ]:
# Puede tardar varios minutos en 83k estudios
enriched_inclusion = []
enriched_exclusion = []

total      = len(df_elig)
checkpoint = total // 10

for i, (_, row) in enumerate(df_elig.iterrows()):
    incl = row['inclusion_criteria']
    excl = row['exclusion_criteria']

    enriched_inclusion.append(enrich_criteria_list(list(incl) if hasattr(incl, '__iter__') else []))
    enriched_exclusion.append(enrich_criteria_list(list(excl) if hasattr(excl, '__iter__') else []))

    if (i + 1) % checkpoint == 0:
        print(f'  {i + 1:>6,} / {total:,} ({(i+1)/total:.0%})')

print(f'\nTotal procesados: {total:,}')


### 2.7 Inspección de resultados

In [ ]:
sample_idx = df_elig[df_elig['n_inclusion'] >= 3].index[0]
nct = df_elig.loc[sample_idx, 'nct_id']

print(f'=== {nct} ===\n')
print('INCLUSIÓN original:')
for c in list(df_elig.loc[sample_idx, 'inclusion_criteria'])[:3]:
    print(f'  - {c}')

print('\nINCLUSIÓN enriquecida:')
for c in enriched_inclusion[sample_idx][:3]:
    print(f'  - {c}')

print('\nEXCLUSIÓN original:')
for c in list(df_elig.loc[sample_idx, 'exclusion_criteria'])[:3]:
    print(f'  - {c}')

print('\nEXCLUSIÓN enriquecida:')
for c in enriched_exclusion[sample_idx][:3]:
    print(f'  - {c}')


### 2.8 Guardado

In [ ]:
df_elig_out = pd.DataFrame({
    'nct_id':             df_elig['nct_id'].values,
    'enriched_inclusion': enriched_inclusion,
    'enriched_exclusion': enriched_exclusion,
})

df_elig_out.to_parquet(PARQUET_OUT_ELIG, index=False)

print(f'Guardado: {PARQUET_OUT_ELIG}')
print(f'Filas: {len(df_elig_out):,}')
print(f'Criterios inclusión totales: {df_elig_out["enriched_inclusion"].apply(len).sum():,}')
print(f'Criterios exclusión totales: {df_elig_out["enriched_exclusion"].apply(len).sum():,}')
df_elig_out.head(3)
